In [5]:
import numpy as np 
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')
nltk.download('ktinker')
import re 
import string
import os
path= '../data/questions.csv'
df=pd.read_csv(path)

[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>
[nltk_data] Error loading ktinker: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>


In [6]:
df=df.reset_index(drop=True)

In [7]:
device='cuda'

In [8]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [9]:
# df=df.drop(columns=['id','qid1','qid2'])
df=df.dropna()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 404348 entries, 0 to 404350
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            404348 non-null  int64 
 1   qid1          404348 non-null  int64 
 2   qid2          404348 non-null  int64 
 3   question1     404348 non-null  object
 4   question2     404348 non-null  object
 5   is_duplicate  404348 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 21.6+ MB


In [10]:
def icon1(l):
    p=str(l).replace(",",' ,') 
    p=p.replace("?"," ?")
    p=p.replace("/"," ") 
    p=p.replace("."," . ")
    p=p.replace('-',' - ')
    p=p.replace("("," ( ") 
    p=p.replace(")"," ) ") 
    p=p.replace("<"," < ") 
    p=p.replace(">"," > ") 
    p=p.replace("\""," \" ") 
    p=p.replace("\'"," \' ")
    p=re.sub(r'[^a-zA-Z\s]','',p)
    p=p.lower()
    return p.split()

In [11]:
df['transformation1']=df['question1'].apply(icon1)
df['transformation2']=df['question2'].apply(icon1)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 404348 entries, 0 to 404350
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               404348 non-null  int64 
 1   qid1             404348 non-null  int64 
 2   qid2             404348 non-null  int64 
 3   question1        404348 non-null  object
 4   question2        404348 non-null  object
 5   is_duplicate     404348 non-null  int64 
 6   transformation1  404348 non-null  object
 7   transformation2  404348 non-null  object
dtypes: int64(4), object(4)
memory usage: 27.8+ MB


In [13]:
vocab=set()
for i in df.question1:
    p=icon1(str(i))
    for j in p:
        vocab.add(j)
for i in df.question2:
    p=icon1(str(i))
    for j in p:
        vocab.add(j)
vocab=list(vocab)

In [14]:
len(vocab)

80808

In [15]:
vocab[:10]

['serology',
 'picsart',
 'shinden',
 'mutliple',
 'tianjin',
 'heinous',
 'enterprenuer',
 'kerr',
 'diagonal',
 'mirae']

In [16]:
id2voc={}
voc2id={}
idx=0
for i in vocab:
    id2voc[idx]=i
    voc2id[i]=idx
    idx+=1
id2voc[idx]='<UNK>'
voc2id['<UNK>']=idx


In [17]:
len(voc2id)

80809

In [219]:
import torch
import torch.nn as nn
vocab_size=len(voc2id)
embedding_dim=728

In [225]:
E=nn.Embedding(vocab_size,embedding_dim).to(device)

In [226]:
sample=df.question1[0]
sample=icon1(sample)
sample_test=[voc2id[i] for i in sample]
sample_test

[63111,
 33193,
 25493,
 55305,
 80200,
 55305,
 43110,
 23156,
 50310,
 54765,
 68688,
 39431,
 54765,
 34703]

In [227]:
x=torch.tensor(sample_test)
embed=E(x.to(device))
print(embed.shape)

torch.Size([14, 728])


In [228]:
avg=embed.mean(dim=0)
avg.shape

torch.Size([728])

In [229]:
def floss(i1,i2):
    return nn.functional.cosine_similarity(i1,i2)

class Embeds(nn.Module):
    def __init__(self,voc2id,E):
        super().__init__()
        self.voc2id=voc2id
        self.E=E
    def embedding(self,x):
        x=[self.voc2id.get(i,self.voc2id['<UNK>']) for i in x]
        return self.E(torch.tensor(x, dtype=torch.long).to(device))
    def forward(self,x):
        emb=self.embedding(x)
        pooled=emb.mean(dim=0)
        return pooled

In [230]:
model=Embeds(voc2id=voc2id,E=E).to(device)


In [286]:
model.eval()
x="what is the capital of china"
y="what is the problem with the neurons in our brain"
op=model(x)
op1=model(y)
op,op.shape

(tensor([ 4.0883e-01,  1.8626e-01,  1.5783e-01,  1.1331e-01,  2.6823e-01,
          3.4722e-02,  5.8490e-01,  2.7502e-01,  2.0270e-01,  2.2105e-01,
          1.5494e-02, -4.4661e-01,  3.7800e-01,  5.9429e-01,  6.7072e-02,
         -4.2804e-01,  2.1091e-01, -1.8763e-02, -1.2021e-01, -9.5863e-01,
         -1.3326e-01, -4.2123e-01,  4.1544e-01,  3.4608e-01,  2.4212e-02,
          2.6443e-01, -8.5270e-02,  4.8937e-02,  3.5334e-01,  1.1703e-01,
          8.7221e-02,  4.9237e-01, -2.6836e-02, -3.9911e-01, -1.2987e-01,
          2.8589e-01, -2.3624e-01,  3.9387e-01,  9.9265e-02,  7.9470e-02,
          3.7252e-01,  1.4163e-01,  2.3951e-01,  3.1594e-01,  2.0648e-01,
         -1.2520e-02, -2.4277e-01,  3.9332e-01,  1.3674e-01, -2.5569e-01,
         -3.7467e-01, -6.4599e-01,  2.3548e-01, -8.4127e-01,  3.9537e-01,
          8.3438e-02, -7.2604e-02,  1.8561e-03,  7.0452e-01,  4.9319e-01,
         -4.3145e-01, -1.5955e-01, -3.2882e-01, -4.6136e-01, -7.9757e-02,
         -2.5967e-01,  1.5178e-01, -2.

In [287]:
similarit=nn.functional.cosine_similarity(
    op1.unsqueeze(0),
    op.unsqueeze(0)
)
similarit

tensor([0.8466], device='cuda:0', grad_fn=<SumBackward1>)

In [278]:
df.loc[df['is_duplicate']==0,'is_duplicate']=-1

In [273]:
import torch.optim as optim
lossfn=nn.CosineEmbeddingLoss()
opt=optim.Adam(model.parameters(),lr=0.05)

In [29]:
def valid_sentence(sentence):
    return len(sentence) > 0

In [91]:
pairs=[]
removed=0
for i in range(len(df)):
    q1 = df.transformation1.iloc[i]
    q2 = df.transformation2.iloc[i]
    label = df.is_duplicate.iloc[i]
    if valid_sentence(q1) and valid_sentence(q2):
        pairs.append((q1, q2, label))
    else:
        print(removed)
        removed += 1
print("Removed:", removed)
print("Remaining:", len(pairs))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
Removed: 28
Remaining: 404320


In [31]:
#model.state_dict=torch.load('embed_model.pth')
model=torch.load('embed_mod.pth',weights_only=False)

In [92]:
pairs[0]

(['what',
  'is',
  'the',
  'step',
  'by',
  'step',
  'guide',
  'to',
  'invest',
  'in',
  'share',
  'market',
  'in',
  'india'],
 ['what',
  'is',
  'the',
  'step',
  'by',
  'step',
  'guide',
  'to',
  'invest',
  'in',
  'share',
  'market'],
 np.int64(-1))

In [233]:
model.E

Embedding(80809, 728)

In [113]:
epoch=1
model.train()
for e in range(epoch):
    total_loss=0
    c=0
    for s1,s2,label in pairs:
        e1=model(s1)
        e2=model(s2)
        target=torch.tensor(label).float().to(device)
        loss=lossfn(e1.unsqueeze(0),e2.unsqueeze(0),target.unsqueeze(0))
        opt.zero_grad()
        loss.backward()
        opt.step()
        c+=1
        total_loss+=loss.item()
        if c%100000==0:
            torch.save(model,"embed_mod.pth")
            print(f'saved:{e}:total loss:{total_loss}')
    print(f'epoch {e} : {total_loss}')

saved:0:total loss:24539.329069257787
saved:0:total loss:53235.03003837455
saved:0:total loss:81774.76825078779
saved:0:total loss:110354.21322563471
epoch 0 : 111590.86722916082


In [178]:
e1.device

device(type='cuda', index=0)

In [64]:
import torch
import pickle
torch.save(model.state_dict(), "embed_model.pth")
with open("voc2id.pkl", "wb") as f:
    pickle.dump(voc2id,f)
with open("id2voc.pkl","wb") as f:
    pickle.dump(id2voc,f)

In [235]:
#batch upgrade
from torch.utils.data import Dataset,DataLoader
class pairDataset(Dataset):
    def __init__(self,pairs):
        self.pairs=pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,idx):
        s1,s2,label=self.pairs[idx]
        return s1,s2,label

In [236]:
ds[0]

(['what',
  'is',
  'the',
  'step',
  'by',
  'step',
  'guide',
  'to',
  'invest',
  'in',
  'share',
  'market',
  'in',
  'india'],
 ['what',
  'is',
  'the',
  'step',
  'by',
  'step',
  'guide',
  'to',
  'invest',
  'in',
  'share',
  'market'],
 np.int64(-1))

In [237]:

with open("voc2id.pkl", "rb") as f:
    voc2id=pickle.load(f)
    print(voc2id['<UNK>'])

80808


In [238]:
from torch.nn.utils.rnn import pad_sequence
import pickle
def collate_fn(batch):
    s1b=[]
    s2b=[]
    label=[]
    with open("voc2id.pkl", "rb") as f:
        voc2id=pickle.load(f)

    for s1,s2,labels in batch:
        ids1=torch.tensor([voc2id[i] for i in s1],dtype=torch.long)
        ids2=torch.tensor([voc2id[i] for i in s2],dtype=torch.long)
        s1b.append(ids1)
        s2b.append(ids2)
        label.append(labels)
    s1b=pad_sequence(s1b,batch_first=True,padding_value=voc2id["<UNK>"])
    s2b=pad_sequence(s1b,batch_first=True,padding_value=voc2id["<UNK>"])
    labels=torch.tensor(labels).float()
    return s1b,s2b,label

In [269]:
ds=pairDataset(pairs)
loader=DataLoader(ds,batch_size=64,shuffle=True,collate_fn=collate_fn)

In [283]:
next(iter(loader))[1]

tensor([[30463, 58845, 56105,  ..., 80808, 80808, 80808],
        [30463, 58845, 56105,  ..., 80808, 80808, 80808],
        [30463, 14933, 56105,  ..., 80808, 80808, 80808],
        ...,
        [63111, 33193,   693,  ..., 80808, 80808, 80808],
        [63111, 33193,  8960,  ..., 80808, 80808, 80808],
        [63111, 54475, 59223,  ..., 80808, 80808, 80808]])

In [272]:
target=torch.tensor(next(iter(loader))[2]).float().to(device)
target.view(-1)

tensor([-1.,  1., -1.,  1., -1., -1., -1.,  1.,  1.,  1., -1., -1., -1., -1.,
         1., -1.,  1., -1.,  1., -1.,  1., -1., -1.,  1.,  1., -1., -1., -1.,
        -1., -1., -1.,  1., -1.,  1., -1., -1.,  1., -1., -1., -1., -1., -1.,
         1., -1.,  1.,  1., -1.,  1.,  1.,  1., -1., -1.,  1., -1., -1.,  1.,
         1.,  1.,  1., -1., -1.,  1., -1.,  1.], device='cuda:0')

In [285]:
epoch=200
model.train()
for e in range(epoch):
    total_loss=0
    for s1,s2,label in loader:
        opt.zero_grad()
        e1=model(s1)
        e2=model(s2)
        target=torch.tensor(label).float().to(device)
        loss=lossfn(e1.unsqueeze(0),e2.unsqueeze(0),target.view(-1))
        loss.backward()
        opt.step()
        total_loss+=loss.item()
    print(f'epoch {e} : {total_loss}')

epoch 0 : 3984.84375
epoch 1 : 3984.9375
epoch 2 : 3984.953125
epoch 3 : 3984.90625
epoch 4 : 3984.890625
epoch 5 : 3984.90625
epoch 6 : 3984.90625
epoch 7 : 3984.9375
epoch 8 : 3984.890625
epoch 9 : 3984.921875
epoch 10 : 3984.859375
epoch 11 : 3984.9375
epoch 12 : 3984.890625
epoch 13 : 3984.9375
epoch 14 : 3984.890625
epoch 15 : 3984.953125
epoch 16 : 3984.890625
epoch 17 : 3984.875
epoch 18 : 3984.921875
epoch 19 : 3984.875
epoch 20 : 3984.90625
epoch 21 : 3984.859375
epoch 22 : 3984.9375
epoch 23 : 3984.875
epoch 24 : 3984.84375
epoch 25 : 3984.90625
epoch 26 : 3984.90625
epoch 27 : 3984.859375
epoch 28 : 3984.921875
epoch 29 : 3984.890625
epoch 30 : 3984.984375
epoch 31 : 3984.921875
epoch 32 : 3984.859375
epoch 33 : 3984.875
epoch 34 : 3984.921875
epoch 35 : 3984.96875
epoch 36 : 3984.953125
epoch 37 : 3984.859375
epoch 38 : 3984.9375
epoch 39 : 3984.828125
epoch 40 : 3984.90625


KeyboardInterrupt: 